# Why the injection protocol needs two rounds of syndrome extraction

Companion notebook to *Pauli web of the $\ket{Y}$ state surface code injection*
(arXiv:2501.15566).

Li and Lao–Criger both specify **two full rounds** of parity measurement before the injected
state is used, and both specify a set of plaquettes to post-select on. Neither derives
either choice. This notebook derives both, by computing the space of closed Pauli webs
directly.

**What comes out:**

| | |
|---|---|
| detector space, $R$ rounds | $\dim = 10 + 24(R-1)$ |
| after 1 round | only **10 of 24** checks are usable; the other 14 give random outcomes |
| after 2 rounds | **all 24** checks are usable, via the round-1 $\times$ round-2 comparison |
| the paper's `+1` post-selection set | *exactly* the 10 round-1 detectors |
| initialisation errors | coverage **saturates at round 1** — more rounds never help |
| blind spots | data qubits **4** (the injected $\ket{Y}$) and **9**: neither error type is ever visible |

So the second round is what turns the surface code on; post-selection is what covers the
one thing no number of rounds can reach.

## 0. Setup

Self-contained apart from `pyzx`. The 3D viewer JavaScript comes from
[github.com/kh428/pyzx_3d_viewer](https://github.com/kh428/pyzx_3d_viewer) (Apache-2.0) and is
inlined into the generated HTML, so the produced pages need only a browser.

In [1]:
import itertools, os, random, sys
import numpy as np
from fractions import Fraction

# run from this notebook's own folder, whatever the kernel's cwd happens to be
HERE = os.getcwd()
assert os.path.exists(os.path.join(HERE, 'injection_webs.py')), (
    f'open this notebook from the folder holding injection_webs.py (cwd is {HERE})')
sys.path.insert(0, HERE)

from injection_webs import (build, web_system, nullspace, solve_with, vec_to_web, audit,
                            covers, detectors, XCHECK, ZCHECK, init_kind, versions,
                            brute_force_allowed, rule_allowed)

for k, v in versions().items():
    print(f'  {k:10s} {v}')

  python     3.12.11
  platform   macOS-15.7.4-arm64-arm-64bit
  numpy      2.4.6
  pyzx       0.10.5


## 1. The rule, checked against the spider tensors

Closed Pauli webs are the null space of a linear system over $\mathbb{F}_2$. The system
encodes one rule: at every spider the decoration must be a stabiliser (or anti-stabiliser) of
that spider. Before trusting the solver, check that rule against the actual tensors.

In [2]:
for colour in ('Z', 'X'):
    for n in (1, 2, 3, 4, 5):
        for ph in (Fraction(0), Fraction(1), Fraction(1, 2), Fraction(-1, 2)):
            assert sorted(brute_force_allowed(colour, n, ph)) == sorted(rule_allowed(colour, n, ph))
print('rule == brute force for Z and X spiders, n = 1..5, phase in {0, pi, +-pi/2}')
print()
print('  the injected corner, a green pi/2 spider (|Y>):', rule_allowed('Z', 1, Fraction(1, 2)))
print('  a |+> initial state, green phase 0            :', rule_allowed('Z', 1, Fraction(0)))
print('  a |0> initial state, red phase 0              :', rule_allowed('X', 1, Fraction(0)))

rule == brute force for Z and X spiders, n = 1..5, phase in {0, pi, +-pi/2}

  the injected corner, a green pi/2 spider (|Y>): ['I', 'Y']
  a |+> initial state, green phase 0            : ['I', 'X']
  a |0> initial state, red phase 0              : ['I', 'Z']


## 2. The diagram

`build(rounds=R)` gives the Lao–Criger initialisation followed by $R$ full rounds (each an
$X$-check layer then a $Z$-check layer), then an open output boundary.

Measurement outcomes are **classical**, so no measurement legs are drawn: the measurement
effect is a one-legged spider of the ancilla's own colour and fuses into it. A check enters a
web exactly when the web covers its ancilla **on every leg**.

In [3]:
print('Lao-Criger initialisation (row 4 = top, column 0 = left):')
for r in range(4, -1, -1):
    print('   ' + ' '.join({'plus': '|+>', 'zero': '|0>', 'Y': '|Y>'}[init_kind(c * 5 + r)]
                           for c in range(5)))
print()
for R in (1, 2):
    g, V, meta = build(rounds=R)
    print(f'  rounds={R}: {g.num_vertices():3d} vertices, {g.num_edges():3d} edges')

Lao-Criger initialisation (row 4 = top, column 0 = left):
   |Y> |0> |0> |0> |0>
   |+> |+> |0> |0> |0>
   |+> |+> |+> |0> |0>
   |+> |+> |+> |+> |0>
   |+> |+> |+> |+> |+>

  rounds=1: 124 vertices, 155 edges
  rounds=2: 198 vertices, 285 edges


## 3. One round: only 10 of the 24 checks are usable

A **detector** is a closed web with no support on the open output boundary: a product of
measurement outcomes that is deterministic. After a single round, a check is a detector only
if its plaquette is already an eigenstate of the initial product state.

In [4]:
g1, V1, m1 = build(rounds=1)
dim1, found1 = detectors(g1, V1, 1)
print(f'detector space dimension after 1 round: {dim1}')

pred = {('X', j) for j, qs in XCHECK.items() if all(init_kind(i) == 'plus' for i in qs)} | \
       {('Z', k) for k, qs in ZCHECK.items() if all(init_kind(i) == 'zero' for i in qs)}
got = {(t, j) for t, r, j in found1}
print(f'  checks covered by a detector : {sorted(got)}')
print(f'  predicted from the init state: {sorted(pred)}')
print(f'  agree: {got == pred}')

detector space dimension after 1 round: 10
  checks covered by a detector : [('X', 0), ('X', 1), ('X', 3), ('X', 4), ('X', 6), ('X', 9), ('Z', 7), ('Z', 9), ('Z', 10), ('Z', 11)]
  predicted from the init state: [('X', 0), ('X', 1), ('X', 3), ('X', 4), ('X', 6), ('X', 9), ('Z', 7), ('Z', 9), ('Z', 10), ('Z', 11)]
  agree: True


### This reproduces the paper's post-selection figure exactly

Figure 11 of the paper marks ten plaquettes `+1`, citing Li/Lao–Criger. Reading those
positions out of the TikZ gives $X$-checks $\{0,1,3,4,6,9\}$ and $Z$-checks $\{7,9,10,11\}$.

In [5]:
paper_plus1 = {('X', j) for j in (0, 1, 3, 4, 6, 9)} | {('Z', k) for k in (7, 9, 10, 11)}
print(f'  paper figure 11 `+1` plaquettes: {sorted(paper_plus1)}')
print(f'  round-1 detectors from solver  : {sorted(got)}')
print(f'\n  identical: {paper_plus1 == got}')
print('\n  -> the post-selection set is not a convention, it is forced:')
print('     it is exactly the set of checks that have a round-1 detector.')

  paper figure 11 `+1` plaquettes: [('X', 0), ('X', 1), ('X', 3), ('X', 4), ('X', 6), ('X', 9), ('Z', 7), ('Z', 9), ('Z', 10), ('Z', 11)]
  round-1 detectors from solver  : [('X', 0), ('X', 1), ('X', 3), ('X', 4), ('X', 6), ('X', 9), ('Z', 7), ('Z', 9), ('Z', 10), ('Z', 11)]

  identical: True

  -> the post-selection set is not a convention, it is forced:
     it is exactly the set of checks that have a round-1 detector.


## 4. The other 14 cells: every one of them, drawn

These are the checks with **no** round-1 detector. Their first-round outcome is uniformly
random, so it cannot be compared against anything. Each still supports a closed web, but that
web runs *forward* to the outputs and touches no initial state, which is precisely the
statement that the measurement creates a stabiliser rather than checking one.

In [6]:
def no_outputs(g, V, xv, zv):
    f = {}
    for i in range(25):
        e = tuple(sorted((V[('out', i)], next(iter(g.neighbors(V[('out', i)]))))))
        f[xv[e]] = 0; f[zv[e]] = 0
    return f

def cover_constraints(g, V, xv, zv, kind, rd, idx):
    v = V[('bx', rd, idx) if kind == 'X' else ('az', rd, idx)]
    var, other = (xv, zv) if kind == 'X' else (zv, xv)
    out = {}
    for w in g.neighbors(v):
        e = tuple(sorted((v, w)))
        out[var[e]] = 1; out[other[e]] = 0
    return out

A1, e1, xv1, zv1, _ = web_system(g1)
rows = []
for kind, dd in (('X', XCHECK), ('Z', ZCHECK)):
    for j, qs in dd.items():
        f = dict(no_outputs(g1, V1, xv1, zv1))
        f.update(cover_constraints(g1, V1, xv1, zv1, kind, 1, j))
        is_det = solve_with(g1, f) is not None
        kinds = [init_kind(i) for i in qs]
        want = 'plus' if kind == 'X' else 'zero'
        wrong = sum(1 for k in kinds if k != want)
        rows.append((kind, j, qs, wrong, is_det))

print(f'{"check":8s} {"support":22s} {"wrong-basis qubits":20s} round-1 detector?')
for kind, j, qs, wrong, is_det in rows:
    print(f'{kind}{j:<7d} {str(qs):22s} {wrong:<20d} {"yes" if is_det else "NO"}')
nbad = sum(1 for *_, d in rows if not d)
print(f'\n{nbad} of 24 checks have no round-1 detector '
      f'(and every one of them has at least one wrong-basis qubit).')

check    support                wrong-basis qubits   round-1 detector?
X0       [0, 1, 5, 6]           0                    yes
X1       [2, 3, 7, 8]           0                    yes
X2       [4, 9]                 2                    NO
X3       [5, 10]                0                    yes
X4       [6, 7, 11, 12]         0                    yes
X5       [8, 9, 13, 14]         3                    NO
X6       [10, 11, 15, 16]       0                    yes
X7       [12, 13, 17, 18]       3                    NO
X8       [14, 19]               2                    NO
X9       [15, 20]               0                    yes
X10      [16, 17, 21, 22]       3                    NO
X11      [18, 19, 23, 24]       4                    NO
Z0       [0, 1]                 2                    NO
Z1       [2, 3]                 2                    NO
Z2       [1, 2, 6, 7]           4                    NO
Z3       [3, 4, 8, 9]           3                    NO
Z4       [5, 6, 10, 11]    

## 5. Two rounds: every check becomes a detector

The second round does not fix the initial state; it gives each check something to be
*compared against*. The round-1 outcome and the round-2 outcome of the same check have a
deterministic product regardless of what the initial state was.

In [7]:
for R in (1, 2, 3, 4):
    g, V, meta = build(rounds=R)
    dim, found = detectors(g, V, R)
    per = {rd: len({(t, j) for t, r, j in found if r == rd}) for rd in range(1, R + 1)}
    print(f'  R={R}: detector dim {dim:3d}  (= 10 + 24(R-1)? {dim == 10 + 24 * (R - 1)})'
          f'   checks covered per round: {per}')
print()
print('  -> round 1 contributes only its 10 deterministic checks.')
print('     every round after the first contributes all 24, as the round-to-round comparison.')
print('     Two rounds is the minimum at which the code checks all of its stabilisers.')

  R=1: detector dim  10  (= 10 + 24(R-1)? True)   checks covered per round: {1: 10}
  R=2: detector dim  34  (= 10 + 24(R-1)? True)   checks covered per round: {1: 24, 2: 24}


  R=3: detector dim  58  (= 10 + 24(R-1)? True)   checks covered per round: {1: 24, 2: 24, 3: 24}


  R=4: detector dim  82  (= 10 + 24(R-1)? True)   checks covered per round: {1: 24, 2: 24, 3: 24, 4: 24}

  -> round 1 contributes only its 10 deterministic checks.
     every round after the first contributes all 24, as the round-to-round comparison.
     Two rounds is the minimum at which the code checks all of its stabilisers.


### Full cells and half cells

The detector space splits by whether a detector reaches back to the initial states. A
**half cell** runs from the initial states up to round 1; a **full cell** closes between two
consecutive rounds and touches neither end of time. Counting them separately shows where the
second round's extra detectors actually come from.

In [8]:
for R in (1, 2, 3):
    g, V, meta = build(rounds=R)
    A, edges, xv, zv, _ = web_system(g)
    f = dict(no_outputs(g, V, xv, zv))
    total = len(solve_with(g, f)[1])
    for i in range(25):                  # additionally forbid support on the init layer
        e = tuple(sorted((V[('init', i)], V[('d', 1, 'x', i)])))
        f[xv[e]] = 0; f[zv[e]] = 0
    full = len(solve_with(g, f)[1])
    print(f'  R={R}: {total:3d} detectors = {full:3d} full cells (no init support) '
          f'+ {total - full:2d} half cells anchored on the initial states')
print()
print('  the half-cell count NEVER grows: every detector that reaches back to the initial')
print('  states was already available after round 1. The second round adds 24 full cells')
print('  per round and not one half cell -- the same conclusion the blind-spot computation')
print('  below reaches from the other direction.')

  R=1:  10 detectors =   0 full cells (no init support) + 10 half cells anchored on the initial states


  R=2:  34 detectors =  24 full cells (no init support) + 10 half cells anchored on the initial states

  R=3:  58 detectors =  48 full cells (no init support) + 10 half cells anchored on the initial states

  the half-cell count NEVER grows: every detector that reaches back to the initial
  states was already available after round 1. The second round adds 24 full cells
  per round and not one half cell -- the same conclusion the blind-spot computation
  below reaches from the other direction.


### The web that rescues `a5`

`a5` (the $Z$-check on qubits 7, 8, 12, 13, three of them $\ket{+}$) has no round-1 detector.
Over two rounds it does: a closed web covering `a5` in *both* rounds, touching neither the
initial states nor the outputs. That is the detector cell the second round buys you.

In [9]:
def web_covering(rounds, targets, detector=True):
    g, V, meta = build(rounds=rounds)
    A, edges, xv, zv, _ = web_system(g)
    f = dict(no_outputs(g, V, xv, zv)) if detector else {}
    for kind, rd, idx in targets:
        f.update(cover_constraints(g, V, xv, zv, kind, rd, idx))
    res = solve_with(g, f)
    if res is None:
        return g, V, meta, None
    part, basis, (ed, xvv, zvv) = res
    best, bw = part.copy(), sum(1 for e in ed if part[xvv[e]] or part[zvv[e]])
    for k in (1, 2):
        for combo in itertools.combinations(range(len(basis)), k):
            v = part.copy()
            for c in combo: v = v ^ basis[c]
            w = sum(1 for e in ed if v[xvv[e]] or v[zvv[e]])
            if w < bw: best, bw = v, w
    rng = random.Random(0)
    for _ in range(20000):
        if not len(basis): break
        v = best ^ basis[rng.randrange(len(basis))]
        w = sum(1 for e in ed if v[xvv[e]] or v[zvv[e]])
        if w < bw: best, bw = v, w
    return g, V, meta, vec_to_web(best, ed, xvv, zvv)

print('a5 as a round-1 detector      :',
      'exists' if web_covering(1, [('Z', 1, 5)])[3] else 'NO SUCH WEB (system inconsistent)')
g2, V2, m2, w2 = web_covering(2, [('Z', 1, 5), ('Z', 2, 5)])
print(f'a5 over two rounds            : {len(w2)} edges, '
      f'{"CLOSED" if not audit(g2, w2, m2) else "violations"}')
outs = [i for i in range(25)
        if tuple(sorted((V2[('out', i)], next(iter(g2.neighbors(V2[('out', i)]))))))in w2]
inits = [i for i in range(25) if tuple(sorted((V2[('init', i)], V2[('d', 1, 'x', i)]))) in w2]
print(f'   touches initial states {inits or "none"}, outputs {outs or "none"}  -> a DETECTOR')

a5 as a round-1 detector      : NO SUCH WEB (system inconsistent)
a5 over two rounds            : 24 edges, CLOSED
   touches initial states none, outputs none  -> a DETECTOR


## 6. What two rounds cannot fix

If the second round makes every check usable, why post-select at all? Because the round-1
$\times$ round-2 detectors are anchored to *neither* end of time: they compare two rounds and
so are blind to anything that went wrong before the first one. Initialisation-error coverage
is therefore fixed by round 1 and never improves.

In [10]:
def init_errors_seen(R):
    g, V, meta = build(rounds=R)
    A, edges, xv, zv, _ = web_system(g)
    part, basis, (ed, xvv, zvv) = solve_with(g, no_outputs(g, V, xv, zv))
    cX, cZ = set(), set()
    for bv in basis:
        w = vec_to_web(bv, ed, xvv, zvv)
        for i in range(25):
            p = w.get(tuple(sorted((V[('init', i)], V[('d', 1, 'x', i)]))), 'I')
            if p in ('Z', 'Y'): cX.add(i)     # an X error anticommutes with Z or Y
            if p in ('X', 'Y'): cZ.add(i)
    return cX, cZ

for R in (1, 2, 3):
    cX, cZ = init_errors_seen(R)
    blind = [i for i in range(25) if i not in cX and i not in cZ]
    print(f'  R={R}: init X error seen on {len(cX):2d}/25 qubits, '
          f'Z error on {len(cZ):2d}/25;  blind spots {blind}')
print()
print('  identical for every R: extra rounds add no initialisation-error coverage at all.')
print()
print('  the two blind spots:')
for i in (4, 9):
    xs = [j for j, qs in XCHECK.items() if i in qs]
    zs = [k for k, qs in ZCHECK.items() if i in qs]
    print(f'    qubit {i} ({init_kind(i)}): in X-checks {xs}, Z-checks {zs}')
print('    none of those checks is a round-1 detector, so an initialisation error')
print('    on the injected corner (4) or its neighbour (9) is invisible. That is what')
print('    post-selection is for, and why the injection fidelity is capped.')

  R=1: init X error seen on  9/25 qubits, Z error on 14/25;  blind spots [4, 9]
  R=2: init X error seen on  9/25 qubits, Z error on 14/25;  blind spots [4, 9]


  R=3: init X error seen on  9/25 qubits, Z error on 14/25;  blind spots [4, 9]

  identical for every R: extra rounds add no initialisation-error coverage at all.

  the two blind spots:
    qubit 4 (Y): in X-checks [2], Z-checks [3]
    qubit 9 (zero): in X-checks [2, 5], Z-checks [3]
    none of those checks is a round-1 detector, so an initialisation error
    on the injected corner (4) or its neighbour (9) is invisible. That is what
    post-selection is for, and why the injection fidelity is capped.


## 7. Interactive 3D viewer

`make_viewers.py` writes a self-contained HTML page with **every detector as its own toggle**.
Pick a graph, then click any number of them to overlay their webs.

* **1 round** — 24 toggles. A green outline means the check has a round-1 detector (10 of 24);
  those ten *are* the half cells. Blue means it has none, and the web shown is the
  forward-only one that runs to the outputs and touches no initial state.
* **2 rounds** — 34 toggles: 24 full cells (round-1 $\times$ round-2) plus the 10
  init-anchored half cells, listed separately. $24 + 10 = 34 = \dim$, so the toggles span the
  whole detector space.

Presets give the 14/10 split on both pages, the half cells on their own, and everything at
once. Selected webs are re-indexed on every rebuild so the viewer's nested tube radii stay
distinct for small selections; "spread webs apart" fans them laterally when several share a
wire. Drag to orbit, scroll to zoom.

In [11]:
import subprocess, sys
print(subprocess.run([sys.executable, 'make_viewers.py'], capture_output=True,
                     text=True).stdout[-900:])

versions: {'python': '3.12.11', 'platform': 'macOS-15.7.4-arm64-arm-64bit', 'numpy': '2.4.6', 'pyzx': '0.10.5'}
round 1: 10 detectors, 14 forward-only  (2s)
round 2: 24 round-1 x round-2 full cells  (7s)
round 2: 10 init-anchored half cells  (10s)
presets: 14 non-detectors, 10 detectors
wrote injection_webs_viewer.html (10s total)



In [12]:
from viewer import inline_view

# Inline 3D, with the viewer JS imported from github.com/kh428/pyzx_3d_viewer over
# jsDelivr -- nothing needs to exist locally. (Pass js='inline' to embed a local copy
# instead.) three.js also comes from a CDN, so this needs the network at view time,
# and it renders only in a live notebook: nbconvert stores the HTML but never runs it.
g1a, V1a, m1a, w_a7 = web_covering(1, [('Z', 1, 7)])                 # a valid round-1 detector
_, _, _, w_a5 = web_covering(1, [('Z', 1, 5)], detector=False)       # a5: forward-only
assert list(g1a.vertices()) == list(build(rounds=1)[0].vertices())   # build() is deterministic

inline_view(g1a, [w_a7, w_a5], node_size=0.16, edge_radius=0.05, web_offset=0.8)

In [13]:
# the two-round detector for a5: closes between the rounds, touching neither end of time
inline_view(g2, w2, node_size=0.16, edge_radius=0.05)

## 8. Summary

1. **$\dim(\text{detectors}) = 10 + 24(R-1)$.** Round 1 contributes only the 10 checks already
   deterministic on the initial product state; every subsequent round contributes all 24.
2. **Two rounds is the minimum** at which the code checks all of its stabilisers. That is the
   derivation of a protocol choice Li and Lao–Criger state without one. Splitting by shape,
   $34 = 24$ full cells (round-1 $\times$ round-2) $+\ 10$ half cells (initial states $\to$
   round 1); the half-cell count never grows, so every detector anchored on the initial
   states was already there after round 1.
3. **The post-selection set is forced**, not conventional: the paper's ten `+1` plaquettes are
   exactly the ten checks that have a round-1 detector.
4. **Rounds and post-selection do different jobs.** Extra rounds give no extra
   initialisation-error coverage — that saturates at round 1. Post-selection is the only thing
   covering initialisation, and even it cannot see data qubits 4 and 9.
5. **The injected corner is a blind spot.** Qubit 4 carries the $\ket{Y}$ and sits in no
   round-1 detector at all, so an initialisation error exactly at the injection site is
   invisible to every web. That is a clean diagrammatic statement of why injection fidelity is
   limited and why the schemes grow the distance afterwards.

Points 1–4 are candidates for the paper. Point 5 is the one I would lead with.